# 03 · EEG features and hypnodensities

Assemble precomputed GSSC/YASA/SleepFM features and hypnodensities using the exact recording–block–epoch–electrode key. Multichannel predictions from another run are not implicitly merged into single-channel features.

## i) Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from dmt_hypnodensities import (
    assemble_outputs, feature_value_columns, fit_mixed_models,
    join_epoch_features_hypnodensities, load_config,
    plot_ranked_stage_features, plot_stage_feature_correlation_heatmap,
    plot_stage_feature_scatter, prepare_run, prepare_treatment_effects,
    prepare_within_condition_changes, save_figure, save_table,
    stage_feature_effect_correlations,
)

## ii) Parameters and persisted run

In [ ]:
CONFIG_PATH = next(path.resolve() for path in (Path('configs/analysis.yaml'), Path('../configs/analysis.yaml')) if path.is_file())
RUN_NAME = 'gap_sensitivity_090s_gssc_yasa_sleepfm_cpu_v1'
STAGES = ('W', 'N1', 'N2', 'N3', 'R')
PROBABILITIES = tuple(f'prob_{stage}' for stage in STAGES)
STAGERS = ('gssc', 'yasa', 'sleepfm')
CONTRAST = 'before_to_after'
DELTA_TYPE = 'abs'
CORRELATION_METHOD = 'spearman'

run = prepare_run(load_config(CONFIG_PATH), RUN_NAME, reuse_existing=True)
tables = assemble_outputs(run.recordings, strict=True)
FEATURES = feature_value_columns(tables.features)
display({'n_features': len(FEATURES), 'features': FEATURES})

## iii) Validated analytical assembly

In [ ]:
epoch_analysis = join_epoch_features_hypnodensities(
    tables.features, tables.hypnodensities, strict=True
)
save_table(epoch_analysis, run.tables / 'epoch_feature_hypnodensity.parquet')
display(epoch_analysis.shape, epoch_analysis.groupby('stager').size())

## iv) Statistics

In [ ]:
joint_changes = prepare_within_condition_changes(
    epoch_analysis,
    value_columns=(*PROBABILITIES, *FEATURES),
    delta_types=('abs', 'rel'),
)
joint_effects = prepare_treatment_effects(
    joint_changes,
    value_columns=(*PROBABILITIES, *FEATURES),
    delta_types=('abs', 'rel'),
)
stage_feature_correlations = stage_feature_effect_correlations(
    joint_effects,
    stage_columns=tuple(f'{column}__effect' for column in PROBABILITIES),
    feature_columns=tuple(f'{column}__effect' for column in FEATURES),
)
feature_mixed_models = fit_mixed_models(
    tables.features,
    outcomes=FEATURES,
    fixed_effects='C(condition) * C(experimental_label)',
    variance_components={'electrode': '0 + C(electrode)'},
    stratify_by=(),
)

for name, table in {
    'joint_changes': joint_changes,
    'joint_treatment_effects': joint_effects,
    'stage_feature_correlations': stage_feature_correlations,
    'feature_mixed_models': feature_mixed_models,
}.items():
    save_table(table, run.tables / f'{name}.csv')
display(stage_feature_correlations, feature_mixed_models)

## v) Figures

In [ ]:
for stager in STAGERS:
    figure, _ = plot_stage_feature_correlation_heatmap(
        stage_feature_correlations,
        stager=stager, contrast=CONTRAST, delta_type=DELTA_TYPE,
        method=CORRELATION_METHOD,
    )
    save_figure(figure, run.figures / f'stage_feature_heatmap_{stager}')
    plt.show()

    figure, _ = plot_ranked_stage_features(
        stage_feature_correlations, stage='R', stager=stager,
        contrast=CONTRAST, delta_type=DELTA_TYPE,
        method=CORRELATION_METHOD, top_n=20,
    )
    save_figure(figure, run.figures / f'R_feature_ranking_{stager}')
    plt.show()

In [ ]:
selected = stage_feature_correlations.loc[
    stage_feature_correlations['stager'].eq(STAGERS[0])
    & stage_feature_correlations['contrast'].eq(CONTRAST)
    & stage_feature_correlations['delta_type'].eq(DELTA_TYPE)
    & stage_feature_correlations['method'].eq(CORRELATION_METHOD)
    & stage_feature_correlations['stage'].eq('R')
].dropna(subset=['correlation']).copy()
selected['absolute_correlation'] = selected['correlation'].abs()
TOP_FEATURE = selected.nlargest(1, 'absolute_correlation')['feature'].iloc[0]
figure, _ = plot_stage_feature_scatter(
    epoch_analysis.loc[epoch_analysis['stager'].eq(STAGERS[0])],
    stage='R', feature=TOP_FEATURE,
)
save_figure(figure, run.figures / f'R_{TOP_FEATURE}_scatter_{STAGERS[0]}')
plt.show()